In [1]:
import os
import time
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from copy import deepcopy

import gr00t
from gr00t.data.dataset.lerobot_episode_loader import LeRobotEpisodeLoader
from gr00t.data.dataset.sharded_single_step_dataset import extract_step_data
from gr00t.data.embodiment_tags import EmbodimentTag
from gr00t.policy.gr00t_policy import Gr00tPolicy

# Import set_seed for reproducibility across PyTorch and TensorRT benchmarks
from scripts.deployment.standalone_inference_script import set_seed
set_seed(42)

In [2]:
# Configuration
MODEL_PATH = "/home/diden/.cache/huggingface/hub/models--DidenRobotics--Humanoid-Upper-v1-Teleoperation/snapshots/503d1102b0d80532aa491442b43043c660fb7d5c/groot_ckpt"

REPO_PATH = os.path.dirname(os.path.dirname(gr00t.__file__))
DATASET_PATH = os.path.join(REPO_PATH, "/home/diden/Projects3/DIDEN_Core/dataset/push_buttons_valid")
EMBODIMENT_TAG = "new_embodiment"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [3]:
policy = Gr00tPolicy(
    model_path=MODEL_PATH,
    embodiment_tag=EmbodimentTag(EMBODIMENT_TAG),
    device=device,
    strict=True,
)

print(f"Model loaded successfully!")
print(f"Action horizon: {policy.model.action_head.action_horizon}")
print(f"Num inference timesteps (denoising steps): {policy.model.action_head.num_inference_timesteps}")

/home/diden/Projects3/DIDEN_Core/diden_vla/Isaac-GR00T/.venv/lib/python3.10/site-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in Qwen3VLForConditionalGeneration is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `dtype` argument. Example: `model = AutoModel.from_pretrained("openai/whisper-tiny", attn_implementation="flash_attention_2", dtype=torch.float16)`
Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in Qwen3VLModel is torch.float32. You should run training or inference using

Total number of DiT parameters:  1091722240
Total number of SelfAttentionTransformer parameters:  201433088


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model loaded successfully!
Action horizon: 40
Num inference timesteps (denoising steps): 4


In [5]:
# Get modality config from policy
modality_config = policy.get_modality_config()
print("Modality config keys:", modality_config.keys())

# Load dataset
dataset = LeRobotEpisodeLoader(
    dataset_path=DATASET_PATH,
    modality_configs=modality_config,
    video_backend="torchcodec",
    video_backend_kwargs=None,
)
print(f"Dataset loaded with {len(dataset)} episodes")

Modality config keys: dict_keys(['video', 'state', 'action', 'language'])
Dataset loaded with 16 episodes


In [6]:
# Print detailed modality config
print("Modality Configuration Details:")
print("=" * 60)
for modality_name, config in modality_config.items():
    print(f"\n{modality_name.upper()}:")
    print(f"  delta_indices: {config.delta_indices}")
    print(f"  modality_keys: {config.modality_keys}")
    if config.sin_cos_embedding_keys:
        print(f"  sin_cos_embedding_keys: {config.sin_cos_embedding_keys}")
    if config.action_configs:
        print(f"  action_configs:")
        for i, ac in enumerate(config.action_configs):
            print(f"    [{i}] {config.modality_keys[i]}: rep={ac.rep.value}, type={ac.type.value}")

Modality Configuration Details:

VIDEO:
  delta_indices: [0]
  modality_keys: ['ego_cam']

STATE:
  delta_indices: [0]
  modality_keys: ['joint_pos', 'joint_vel', 'ee_pos']

ACTION:
  delta_indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
  modality_keys: ['joint_pos']
  action_configs:
    [0] joint_pos: rep=relative, type=non_eef

LANGUAGE:
  delta_indices: [0]
  modality_keys: ['annotation.human.task_description']


In [ ]:
# Extract a sample step data
episode_data = dataset[0]
step_data = extract_step_data(
    episode_data, 
    step_index=0, 
    modality_configs=modality_config, 
    embodiment_tag=EmbodimentTag(EMBODIMENT_TAG), 
    allow_padding=False
)

# Prepare observation in the format expected by the policy
observation = {
    "video": {k: np.stack(step_data.images[k])[None] for k in step_data.images},
    "state": {k: step_data.states[k][None] for k in step_data.states},
    "action": {k: step_data.actions[k][None] for k in step_data.actions},
    "language": {
        modality_config["language"].modality_keys[0]: [[step_data.text]],
    }
}

print("\nObservation shapes:")
print(f"  Video keys: {list(observation['video'].keys())}")
for k, v in observation['video'].items():
    print(f"    {k}: {v.shape}")
print(f"  State keys: {list(observation['state'].keys())}")
for k, v in observation['state'].items():
    print(f"    {k}: {v.shape}")
print(f"  Action keys: {list(observation['action'].keys())}")
for k, v in observation['action'].items():
    print(f"    {k}: {v.shape}")
print(f"  Language: {observation['language']}")